# Work Units and Integral Buffers

This notebook explores LibAccInt's low-level **ShellSetPair** and **ShellSetQuartet** APIs
for fine-grained integral computation.

**What you'll learn:**
- How shells are grouped into `ShellSet` objects by (angular momentum, contraction degree)
- Using `ShellSetPair` and `engine.compute()` for one-electron integrals
- Using `ShellSetQuartet` and `engine.compute()` for two-electron integrals
- Scattering `IntegralBuffer` data into full matrices using `pair_meta` / `quartet_meta`
- The compute-and-consume pattern for Fock matrix construction

In [ ]:
import numpy as np
import libaccint as lai

print(f"LibAccInt {lai.__version__}")
print(f"CUDA: {lai.has_cuda_backend()}")

## 1. Setup

We use hydrogen fluoride (HF) with the 6-31G\*\* basis. This is small enough to inspect
every work unit while still demonstrating multiple angular momentum types.

In [ ]:
# HF molecule: F-H bond length = 1.7328 Bohr (0.9169 Ang)
atoms = [
    lai.Atom(9, [0.0, 0.0, 0.0]),       # Fluorine
    lai.Atom(1, [0.0, 0.0, 1.7328]),     # Hydrogen
]

basis = lai.basis_set("6-31G**", atoms)
engine = lai.Engine(basis)
nbf = basis.n_basis_functions()

print(f"HF / 6-31G**: {nbf} basis functions, {basis.n_shells()} shells")
print(f"GPU available: {engine.gpu_available()}")

## 2. ShellSets

Shells with the same angular momentum and contraction degree are grouped into **ShellSets**.
This batching is key to GPU efficiency — kernels are specialized per AM type.

In [ ]:
am_labels = ['s', 'p', 'd', 'f', 'g']

# Discover shell set keys from individual shells
shell_set_keys = set()
for i in range(basis.n_shells()):
    s = basis.shell(i)
    shell_set_keys.add((s.angular_momentum(), s.n_primitives()))

print(f"{len(shell_set_keys)} unique ShellSets (grouped by AM and contraction degree):")
for am, k in sorted(shell_set_keys):
    ss = basis.shell_set(am, k)
    print(f"  {am_labels[am]}-type, K={k}: {ss.n_shells()} shells, "
          f"{ss.n_functions_per_shell()} functions/shell")

## 3. ShellSetPairs for One-Electron Integrals

A `ShellSetPair` combines two `ShellSet` objects. Each pair defines a batch of
shell-pair integrals with the same angular momentum pattern $(l_a, l_b)$.

In [ ]:
pairs = basis.shell_set_pairs()
print(f"{len(pairs)} ShellSetPairs:")
for i, pair in enumerate(pairs):
    print(f"  Pair {i}: ({am_labels[pair.La()]},{am_labels[pair.Lb()]}) "
          f"- {pair.n_pairs()} shell pairs")

## 4. Computing Integrals per Pair

`engine.compute(op, pair)` is the unified entry point — the engine routes work to
the best available backend (CPU or GPU) based on the operator and work unit type.
It returns an `IntegralBuffer`. We use `pair_meta(idx)` to find where each shell
pair's integrals belong in the full matrix.

In [ ]:
# Assemble the overlap matrix from work units
# engine.compute() accepts an optional BackendHint (defaults to Auto)
S_work = np.zeros((nbf, nbf))
op_overlap = lai.Operator.overlap()

for pair in pairs:
    buf = engine.compute(op_overlap, pair)  # hint=Auto
    data = buf.to_numpy()

    for idx in range(buf.n_shell_pairs()):
        meta = buf.pair_meta(idx)
        for a in range(meta.na):
            for b in range(meta.nb):
                val = data[meta.offset + a * meta.nb + b]
                S_work[meta.fi + a, meta.fj + b] = val
                S_work[meta.fj + b, meta.fi + a] = val  # symmetry

In [ ]:
# Verify against the direct matrix API
S_direct = engine.compute_overlap_matrix()
max_err = np.max(np.abs(S_work - S_direct))

print(f"Work-unit assembled S matches direct API: {np.allclose(S_work, S_direct)}")
print(f"Max |error|: {max_err:.2e}")
assert np.allclose(S_work, S_direct), "Mismatch!"

## 5. ShellSetQuartets for Two-Electron Integrals

A `ShellSetQuartet` combines a bra pair and ket pair for two-electron integrals
$(\mu\nu|\lambda\sigma)$. Each quartet handles a specific $(l_a, l_b, l_c, l_d)$ pattern.

In [ ]:
quartets = basis.shell_set_quartets()
total_shell_quartets = sum(q.n_quartets() for q in quartets)

print(f"{len(quartets)} ShellSetQuartets, {total_shell_quartets} total shell quartets:")
for i, q in enumerate(quartets):
    print(f"  Quartet {i}: ({am_labels[q.La()]},{am_labels[q.Lb()]}|"
          f"{am_labels[q.Lc()]},{am_labels[q.Ld()]}) - {q.n_quartets()} quartets")

## 6. Computing ERI Integrals per Quartet

`engine.compute(op, quartet)` returns an `IntegralBuffer` — the same unified API,
now dispatching two-electron work to CPU or GPU based on the `ShellSetQuartet`.

We scatter the data into the full ERI tensor using `quartet_meta(idx)`.
Since work units only compute unique quartets, we must apply 8-fold permutation symmetry
$(\mu\nu|\lambda\sigma) = (\nu\mu|\lambda\sigma) = \ldots = (\sigma\lambda|\nu\mu)$
to fill the complete tensor.

In [ ]:
# Assemble the full ERI tensor from work units
# Work units only compute unique quartets, so we apply 8-fold
# permutation symmetry: (mn|ls) = (nm|ls) = (mn|sl) = ... = (sl|nm)
eri_work = np.zeros((nbf, nbf, nbf, nbf))
op_coulomb = lai.Operator.coulomb()

for quartet in quartets:
    buf = engine.compute(op_coulomb, quartet)
    if buf.empty():
        continue
    data = buf.to_numpy()

    for idx in range(buf.n_shell_quartets()):
        meta = buf.quartet_meta(idx)
        for a in range(meta.na):
            for b in range(meta.nb):
                for c in range(meta.nc):
                    for d in range(meta.nd):
                        val = data[meta.offset + ((a * meta.nb + b) * meta.nc + c) * meta.nd + d]
                        m, n, l, s = meta.fi+a, meta.fj+b, meta.fk+c, meta.fl+d
                        # 8-fold permutation symmetry
                        for p, q, r, t in [
                            (m,n,l,s), (n,m,l,s), (m,n,s,l), (n,m,s,l),
                            (l,s,m,n), (s,l,m,n), (l,s,n,m), (s,l,n,m)]:
                            eri_work[p,q,r,t] = val

print(f"Assembled ERI tensor: shape {eri_work.shape}")
print(f"Non-zero elements: {np.count_nonzero(eri_work)}")

In [ ]:
# Verify against the direct ERI tensor API
eri_direct = engine.compute_eri_tensor()
max_err = np.max(np.abs(eri_work - eri_direct))

print(f"Work-unit assembled ERI matches direct API: {np.allclose(eri_work, eri_direct)}")
print(f"Max |error|: {max_err:.2e}")
assert np.allclose(eri_work, eri_direct), "Mismatch!"

## 7. Verifying Against the Fock Builder

We can verify our assembled ERI tensor by comparing Coulomb and exchange matrices
built from it (via `np.einsum`) against those from the `FockBuilder` + `engine.compute()` API.

When `engine.compute(op, fock_builder)` is called with a `FockBuilder` as consumer,
the engine computes all two-electron integrals for the full basis and accumulates
$J$ and $K$ on-the-fly — no $O(N^4)$ tensor storage required.

$$J_{\mu\nu} = \sum_{\lambda\sigma} D_{\lambda\sigma} (\mu\nu|\lambda\sigma), \qquad K_{\mu\nu} = \sum_{\lambda\sigma} D_{\lambda\sigma} (\mu\lambda|\nu\sigma)$$

In [ ]:
# Build J and K from our assembled ERI tensor using einsum
D = np.ascontiguousarray(np.eye(nbf), dtype=np.float64)

J_einsum = np.einsum('ls,mnls->mn', D, eri_work)
K_einsum = np.einsum('ls,mlns->mn', D, eri_work)

# Compare to FockBuilder + engine.compute(op, consumer)
fock = lai.FockBuilder(nbf)
fock.set_density(D)
engine.compute(op_coulomb, fock)
J_fock = fock.get_coulomb_matrix()
K_fock = fock.get_exchange_matrix()

print(f"J (einsum vs FockBuilder): max |err| = {np.max(np.abs(J_einsum - J_fock)):.2e}")
print(f"K (einsum vs FockBuilder): max |err| = {np.max(np.abs(K_einsum - K_fock)):.2e}")
assert np.allclose(J_einsum, J_fock), "J mismatch!"
assert np.allclose(K_einsum, K_fock), "K mismatch!"
print("Both match — our work-unit assembled ERI tensor is correct!")

## Summary

| API | Purpose |
|-----|--------|
| `basis.shell_set_pairs()` | Get all 1e work units |
| `basis.shell_set_quartets()` | Get all 2e work units |
| `engine.compute(op, pair)` | Compute 1e integrals for a ShellSetPair |
| `engine.compute(op, quartet)` | Compute 2e integrals for a ShellSetQuartet |
| `engine.compute(op, consumer)` | Compute full-basis 2e and accumulate into FockBuilder |
| `IntegralBuffer.to_numpy()` | Extract integral values |
| `buf.pair_meta(idx)` / `buf.quartet_meta(idx)` | Get scatter indices |

`engine.compute()` is the unified entry point — the engine automatically routes
work to CPU or GPU based on the operator, work unit type, and `BackendHint`.

**When to use work units:** Custom integral processing, selective computation, or
understanding the engine's internal batching. For standard matrix or Fock builds,
prefer the high-level APIs.

**Next:** [03_fock_matrix_and_screening.ipynb](03_fock_matrix_and_screening.ipynb) — Fock matrix construction and Schwarz screening.